In [7]:
import pandas as pd

# Имена колонок для news.tsv
news_cols = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities']
news_df = pd.read_csv('../data/news.tsv', sep='\t', names=news_cols)

# Имена колонок для behaviors.tsv
behav_cols = ['impression_id', 'user_id', 'time', 'history', 'impressions']
behav_df = pd.read_csv('../data/behaviors.tsv', sep='\t', names=behav_cols)

print(f"Всего новостей: {len(news_df)}")
print(f"Всего показов: {len(behav_df)}")

Всего новостей: 51282
Всего показов: 156965


In [8]:
print("=== News Sample ===")
display(news_df.head(3))

print("\n=== Behaviors Sample ===")
display(behav_df.head(3))

# 3. Проверка пропусков
print("\n=== Пропуски в news_df ===")
print(news_df.isnull().sum())

print("\n=== Пропуски в behav_df ===")
print(behav_df.isnull().sum())

# 4. Уникальные пользователи и новости
unique_users = behav_df['user_id'].nunique()
unique_news_in_catalog = news_df['news_id'].nunique()

print(f"\n Всего записей новостей в каталоге: {len(news_df)} (Уникальных ID: {unique_news_in_catalog})")
print(f" Всего показов (impressions): {len(behav_df)}")
print(f" Уникальных пользователей: {unique_users}")

=== News Sample ===


,news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."



=== Behaviors Sample ===


,impression_id,user_id,time,history,impressions
0,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
1,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...
2,3,U73700,11/14/2019 7:01:48 AM,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...



=== Пропуски в news_df ===
news_id                 0
category                0
subcategory             0
title                   0
abstract             2666
url                     0
title_entities          3
abstract_entities       4
dtype: int64

=== Пропуски в behav_df ===
impression_id       0
user_id             0
time                0
history          3238
impressions         0
dtype: int64

 Всего записей новостей в каталоге: 51282 (Уникальных ID: 51282)
 Всего показов (impressions): 156965
 Уникальных пользователей: 50000


In [9]:
def parse_impressions(imp_str):
    if pd.isna(imp_str):
        return 0, 0
    items = imp_str.split()
    clicks = sum(1 for item in items if item.endswith('-1'))
    total = len(items)
    return clicks, total


# 2. Применяем парсинг
clicks_and_totals = behav_df['impressions'].apply(parse_impressions)
behav_df['clicks_count'] = [x[0] for x in clicks_and_totals]
behav_df['candidates_count'] = [x[1] for x in clicks_and_totals]

# 3. Подсчитываем длину истории
behav_df['history_length'] = behav_df['history'].apply(lambda x: len(str(x).split()) if pd.notna(x) else 0)

# 4. Считаем глобальный CTR
total_clicks = behav_df['clicks_count'].sum()
total_impressions_shown = behav_df['candidates_count'].sum()
global_ctr = (total_clicks / total_impressions_shown) * 100

print("=== Результаты анализа взаимодействий ===")
print(f"Всего показов карточек (Total Impressions): {total_impressions_shown}")
print(f"Всего кликов (Total Clicks): {total_clicks}")
print(f"Глобальный CTR системы: {global_ctr:.2f}%")
print("-" * 40)
print(f"Среднее число кандидатов за 1 сессию: {behav_df['candidates_count'].mean():.1f}")
print(f"Средняя длина истории пользователя: {behav_df['history_length'].mean():.1f} прочитанных статей")
print(f"Максимальная длина истории: {behav_df['history_length'].max()}")

=== Результаты анализа взаимодействий ===
Всего показов карточек (Total Impressions): 5843444
Всего кликов (Total Clicks): 236344
Глобальный CTR системы: 4.04%
----------------------------------------
Среднее число кандидатов за 1 сессию: 37.2
Средняя длина истории пользователя: 32.5 прочитанных статей
Максимальная длина истории: 558


In [10]:
import matplotlib.pyplot as plt
from collections import Counter

# 1. Собираем все кликнутые news_id
all_clicked_news = []
for imp in behav_df['impressions'].dropna():
    items = imp.split()
    for item in items:
        if item.endswith('-1'):
            all_clicked_news.append(item.split('-')[0])

clicked_counts = Counter(all_clicked_news)
click_counts_series = pd.Series(clicked_counts)

# 2. Анализ распределения
total_unique_news_in_catalog = news_df['news_id'].nunique()
clicked_unique_news = len(click_counts_series)
never_clicked_news = total_unique_news_in_catalog - clicked_unique_news

# 3. Сколько кликов берут на себя ТОП-10% самых популярных новостей?
top_10_percent_count = int(clicked_unique_news * 0.1)
clicks_in_top_10_pct = click_counts_series.nlargest(top_10_percent_count).sum()
pct_clicks_top_10 = (clicks_in_top_10_pct / len(all_clicked_news)) * 100

print("=== Анализ Popularity Bias и Cold Start Новости ===")
print(f"Всего уникальных новостей в каталоге: {total_unique_news_in_catalog}")
print(f"Новостей, на которые кликнули хотя бы 1 раз: {clicked_unique_news} ({clicked_unique_news/total_unique_news_in_catalog*100:.1f}%)")
print(f"«Холодных» новостей (0 кликов в логах показов): {never_clicked_news} ({never_clicked_news/total_unique_news_in_catalog*100:.1f}%)")
print("-" * 50)
print(f" Доля кликов, приходящаяся на ТОП-10% популярных новостей: {pct_clicks_top_10:.2f}%")

=== Анализ Popularity Bias и Cold Start Новости ===
Всего уникальных новостей в каталоге: 51282
Новостей, на которые кликнули хотя бы 1 раз: 7713 (15.0%)
«Холодных» новостей (0 кликов в логах показов): 43569 (85.0%)
--------------------------------------------------
 Доля кликов, приходящаяся на ТОП-10% популярных новостей: 78.21%


In [12]:
# 1. Приводим колонку time к типу datetime
behav_df['datetime'] = pd.to_datetime(behav_df['time'], format='%m/%d/%Y %I:%M:%S %p')

min_date = behav_df['datetime'].min()
max_date = behav_df['datetime'].max()
total_days = (max_date - min_date).days

print("=== Временной диапазон логов (Time Drift) ===")
print(f"Старт логов: {min_date}")
print(f"Конец логов: {max_date}")
print(f"Охват в днях: {total_days} дней")

# 2. Смотрим распределение кликов/сессий по дням
behav_df['date'] = behav_df['datetime'].dt.date
daily_sessions = behav_df.groupby('date')['impression_id'].count()

print("\nКоличество сессий показов по дням:")
print(daily_sessions)

=== Временной диапазон логов (Time Drift) ===
Старт логов: 2019-11-09 00:00:19
Конец логов: 2019-11-14 23:59:13
Охват в днях: 5 дней

Количество сессий показов по дням:
date
2019-11-09    13570
2019-11-10    15048
2019-11-11    32799
2019-11-12    33654
2019-11-13    31624
2019-11-14    30270
Name: impression_id, dtype: int64
